# Qwen3.8-27B（原生全精度多模态 API 服务）

- **支持显卡**：A100 (80GB)、H100 (80GB)、RTX 6000 (96GB)（若选 `FP8` 模式，A100 40GB 亦可运行）
- **默认配置**：[`Qwen/Qwen3.8-27B`](https://huggingface.co/Qwen/Qwen3.8-27B) 原生 BF16 全精度（约 55 GB）、**256K（`262144`）原生满血上下文**、原生图像/视频多模态理解、可控深度思考（`reasoning_effort`）与工具调用（Tool Call）
- **双协议原生兼容**：vLLM 在同一 `8188` 端口同时原生暴露 **OpenAI 接口（`/v1/chat/completions`）** 与 **Anthropic Messages 接口（`/v1/messages`）**，无需任何第三方中转即可直连 **Claude Code (`cc`)**、Cursor、Cline、OpenAI SDK
- **执行顺序**：`1. 环境检测` → `2. 安装 vLLM 推理引擎` → `3. 下载 Qwen3.8-27B 模型` → `4. 启动并挂载 API 服务`

| 接口路径 | 协议标准 | 适用客户端 / 支持能力 |
| --- | --- | --- |
| `POST /v1/messages` | Anthropic Messages API | **Claude Code (`cc`)** 直连（官方评测同款 256K 上下文、Thinking 思考块、Tool Use、多模态输入） |
| `POST /v1/chat/completions` | OpenAI Chat Completions | OpenAI SDK、Cursor、NextChat 等（支持文本/图片/视频、快慢思考切换、Function Calling） |
| `GET /v1/models` | OpenAI / Anthropic Models | 模型名支持 `Qwen/Qwen3.8-27B` 或 `qwen3.8-27b` |


### 1. 环境检测
- **功能**：检测 GPU 型号、显存容量、BF16 支持、系统内存及 PyTorch / CUDA 版本。
- **要求**：支持 `A100`、`H100`、`RTX 6000`；跑原生 `BF16` 全精度 + `256K` 满上下文需 80GB 级显存（跑 `FP8` 需 40GB 级及以上）。
- **使用**：直接运行，输出 `[OK]` 后继续。


In [ ]:
# @title 1. 环境检测 {display-mode: "form"}
import os, torch

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU。请在 Colab 菜单选择：代码执行程序 -> 更改运行时类型 -> A100 / H100 / RTX 6000 GPU。")

name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {name} | 显存: {vram_gib:.1f} GiB | 系统内存: {ram_gib:.1f} GiB | BF16: {bf16}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
if not any(k in name.upper() for k in ("A100", "H100", "6000")) or vram_gib < 35:
    raise RuntimeError(f"当前显卡为 {name} ({vram_gib:.1f} GiB)，需要 A100、H100 或 RTX 6000。")
if not bf16:
    raise RuntimeError("当前 GPU 不支持 BF16。")
if vram_gib < 70:
    print("[提示] 当前为 40GB 级显卡，请在步骤 3 将模型精度切换为 FP8。")
print("[OK] GPU 环境符合要求。")


### 2. 安装 vLLM 推理引擎
- **功能**：安装最新版 `vLLM`、HuggingFace 高速下载组件（`hf_xet`）、`openai` 客户端与 `cloudflared`。
- **使用**：直接运行，首次安装约需 1–3 分钟。


In [ ]:
# @title 2. 安装 vLLM 推理引擎 {display-mode: "form"}
import subprocess, sys, urllib.request
from pathlib import Path

def run(args):
    subprocess.run(args, check=True)

run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "vllm",
    "huggingface_hub",
    "hf_xet",
    "openai",
])

cloudflared_bin = Path("/usr/local/bin/cloudflared")
if not cloudflared_bin.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        str(cloudflared_bin),
    )
    cloudflared_bin.chmod(0o755)

vllm_ver = subprocess.check_output([sys.executable, "-c", "import vllm; print(vllm.__version__)"], text=True).strip()
print(f"vLLM: {vllm_ver} | cloudflared: 已就绪")
print("[OK] vLLM 推理环境安装完成。")


### 3. 下载 Qwen3.8-27B 模型
- **功能**：自动读取 Colab Secrets 中的 `HF_TOKEN`（可选），下载官方 `Qwen/Qwen3.8-27B` 原生全精度权重至 `/content/models/`。
- **参数**：
  - `模型精度`：`BF16`（默认，官方原生全精度 `Qwen/Qwen3.8-27B`，约 55 GB，需 80GB/96GB 显卡）或 `FP8`（官方 `Qwen/Qwen3.8-27B-FP8`，约 28 GB，40GB 显卡可用）。
- **使用**：选择精度后运行；已下载分片自动跳过。


In [ ]:
# @title 3. 下载 Qwen3.8-27B 模型 {display-mode: "form"}
模型精度 = "BF16" # @param ["BF16", "FP8"]

import shutil, torch
from pathlib import Path
from huggingface_hub import snapshot_download

vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
if 模型精度 == "BF16" and vram_gib < 70:
    raise RuntimeError(f"当前显存为 {vram_gib:.1f} GiB，不足以加载 55GB 原生 BF16 全精度模型；请将上方「模型精度」切换为 FP8 后重跑。")

MODEL_SPECS = {
    "BF16": {
        "repo_id": "Qwen/Qwen3.8-27B",
        "revision": "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0",
        "local_dir": Path("/content/models/Qwen3.8-27B"),
        "min_free_gb": 58,
    },
    "FP8": {
        "repo_id": "Qwen/Qwen3.8-27B-FP8",
        "revision": "017b9c7af6b5689d5dd426a76e0bc077eb5ca20a",
        "local_dir": Path("/content/models/Qwen3.8-27B-FP8"),
        "min_free_gb": 31,
    },
}

spec = MODEL_SPECS[模型精度]
target_dir = spec["local_dir"]
target_dir.parent.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(target_dir.parent).free / 1024**3
if not (target_dir / "config.json").is_file() and free_gb < spec["min_free_gb"]:
    raise RuntimeError(f"磁盘可用空间仅 {free_gb:.1f} GiB，下载 {spec['repo_id']} 至少需要 {spec['min_free_gb']} GiB。")

print(f"开始同步模型：{spec['repo_id']} -> {target_dir}")
snapshot_download(
    repo_id=spec["repo_id"],
    revision=spec["revision"],
    local_dir=str(target_dir),
)
SELECTED_QWEN38_DIR = str(target_dir)
SELECTED_QWEN38_PRECISION = 模型精度
print(f"[OK] 模型准备完成：{target_dir}")


### 4. 启动并挂载 API 服务
- **功能**：在后台 `127.0.0.1:8188` 启动 `vllm serve`（复用现有的 Cloudflare Tunnel `8188` 端口配置），自动从 Colab Secrets 读取 `CF_TUNNEL_TOKEN`，同时暴露 **OpenAI (`/v1/chat/completions`)** 与 **Anthropic / Claude Code (`/v1/messages`)** 接口。
- **参数**：
  - `MAX_MODEL_LEN`：最大上下文长度（默认直接拉满原生上限 `262144` 即 256K；得益于 Qwen3.8 的 3:1 Gated DeltaNet 线性注意力架构，256K KV Cache 仅占约 16 GiB，单张 80GB/96GB 显卡跑 BF16 即可满血容纳）。
  - `ENABLE_MTP`：开启官方内置 MTP（多 Token 预测）投机采样加速（默认开启）。
  - `RESTART_SERVER`：是否重启已有 vLLM 服务（默认关闭）。
- **使用**：运行后下方会自动输出 **Claude Code (`cc`)** 一键环境变量配置与 OpenAI 调用命令。


In [ ]:
# @title 4. 启动并挂载 API 服务 {display-mode: "form"}
MAX_MODEL_LEN = "262144" # @param ["262144", "131072", "65536"]
ENABLE_MTP = True # @param {type:"boolean"}
RESTART_SERVER = False # @param {type:"boolean"}

import json, os, re, signal, socket, subprocess, sys, time, urllib.request
from pathlib import Path

PORT = 8188
BASE = f"http://127.0.0.1:{PORT}"
LOG_PATH = Path("/content/vllm_qwen38.log")
PID_PATH = Path("/content/vllm_qwen38.pid")
CF_LOG = Path("/content/cloudflared.log")
CF_PID = Path("/content/cloudflared.pid")

model_dir = globals().get("SELECTED_QWEN38_DIR") or "/content/models/Qwen3.8-27B"
if not (Path(model_dir) / "config.json").is_file():
    raise RuntimeError(f"未找到模型目录 {model_dir}，请先运行步骤 3 下载模型。")

def ready():
    try:
        with urllib.request.urlopen(BASE + "/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

def stop_server():
    if PID_PATH.exists():
        try:
            os.kill(int(PID_PATH.read_text().strip()), signal.SIGTERM)
        except Exception:
            pass
    subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"], check=False)
    subprocess.run(["pkill", "-f", "vllm serve"], check=False)
    time.sleep(2)

def resolve_cf_token():
    t = ""
    try:
        from google.colab import userdata
        t = (userdata.get("CF_TUNNEL_TOKEN") or "").strip()
    except Exception:
        t = ""
    if not t:
        t = os.environ.get("CF_TUNNEL_TOKEN", "").strip()
    return t.split()[-1] if t else ""

def start_cf_tunnel(token):
    subprocess.run(["pkill", "-9", "-f", "cloudflared"], check=False)
    if CF_PID.exists():
        CF_PID.unlink(missing_ok=True)
    time.sleep(1)
    cloudflared_bin = Path("/usr/local/bin/cloudflared")
    if not cloudflared_bin.exists():
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            str(cloudflared_bin),
        )
        cloudflared_bin.chmod(0o755)
    with CF_LOG.open("w") as cf_log:
        cf_proc = subprocess.Popen(
            [
                str(cloudflared_bin),
                "tunnel",
                "--no-autoupdate",
                "--protocol", "http2",
                "--edge-ip-version", "4",
                "run",
                "--url", BASE,
                "--token", token,
            ],
            stdin=subprocess.DEVNULL,
            stdout=cf_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    CF_PID.write_text(str(cf_proc.pid))
    deadline = time.time() + 30
    while time.time() < deadline:
        if cf_proc.poll() is not None:
            print(CF_LOG.read_text(errors="replace")[-4000:])
            raise RuntimeError("Cloudflare Tunnel 启动失败，请检查 CF_TUNNEL_TOKEN 是否正确。")
        log_text = CF_LOG.read_text(errors="replace") if CF_LOG.exists() else ""
        if "Registered tunnel connection" in log_text:
            if "Updated to new configuration" not in log_text:
                time.sleep(1)
                log_text = CF_LOG.read_text(errors="replace") if CF_LOG.exists() else ""
            clean_log = log_text.replace('\"', '"')
            routes = re.findall(r'"hostname"\s*:\s*"([^"]+)"\s*,\s*"service"\s*:\s*"([^"]+)"', clean_log)
            if not routes:
                routes = [(h, s) for s, h in re.findall(r'"service"\s*:\s*"([^"]+)"\s*,\s*"hostname"\s*:\s*"([^"]+)"', clean_log)]
            host_url = f"https://{routes[0][0]}" if routes else "<你的CF域名>"
            print(f"[OK] Cloudflare Tunnel 已连接 (HTTP/2): {host_url}")
            return host_url
        time.sleep(1)
    print("[WARN] Cloudflare Tunnel 已在后台运行，详看 /content/cloudflared.log。")
    return "<你的CF域名>"

cf_token = resolve_cf_token()
if not cf_token:
    raise RuntimeError("未检测到 CF_TUNNEL_TOKEN，请在 Colab 左侧 🔑 Secrets 中添加 CF_TUNNEL_TOKEN 并开启访问权限。")

if RESTART_SERVER and ready():
    stop_server()

if ready():
    print("[OK] vLLM 服务已在运行。")
else:
    cmd = [
        "vllm", "serve", model_dir,
        "--served-model-name", "Qwen/Qwen3.8-27B", "qwen3.8-27b",
        "--host", "127.0.0.1",
        "--port", str(PORT),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", "0.93",
        "--enable-prefix-caching",
        "--reasoning-parser", "qwen3",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "qwen3_coder",
        "--trust-remote-code",
    ]
    if ENABLE_MTP:
        cmd.extend(["--speculative-config", '{"method":"mtp","num_speculative_tokens":3}'])
    with LOG_PATH.open("w") as log_fp:
        proc = subprocess.Popen(
            cmd,
            stdin=subprocess.DEVNULL,
            stdout=log_fp,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    PID_PATH.write_text(str(proc.pid))
    print("正在加载 27B 权重并初始化 KV Cache（约需 1~3 分钟）...")
    deadline = time.time() + 900
    while time.time() < deadline:
        if proc.poll() is not None:
            print(LOG_PATH.read_text(errors="replace")[-10000:])
            raise RuntimeError(f"vLLM 提前退出，code={proc.returncode}")
        if ready():
            print("[OK] Qwen3.8-27B vLLM 服务启动成功。")
            break
        time.sleep(3)
    else:
        print(LOG_PATH.read_text(errors="replace")[-10000:])
        raise TimeoutError("vLLM 15 分钟内未就绪。")

host_url = start_cf_tunnel(cf_token)
print(f"""
=== 1. Claude Code (cc) 直连配置（Anthropic /v1/messages 原生协议）===
export ANTHROPIC_BASE_URL="{host_url}"
export ANTHROPIC_AUTH_TOKEN="EMPTY"
export ANTHROPIC_API_KEY="EMPTY"
export ANTHROPIC_MODEL="qwen3.8-27b"
export ANTHROPIC_SMALL_FAST_MODEL="qwen3.8-27b"
export CLAUDE_CODE_DISABLE_NONESSENTIAL_TRAFFIC=1
claude --model qwen3.8-27b

=== 2. OpenAI 兼容接口配置（/v1/chat/completions）===
Base URL : {host_url}/v1
Model    : qwen3.8-27b （或 Qwen/Qwen3.8-27B）
API Key  : EMPTY
""")
